# Modeling: predicting long subway delays (≥ 5 min)

Input: `data/processed/delays_clean.csv`, produced by `01_eda.ipynb`.

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("../data/processed/delays_clean.csv", parse_dates=["datetime"])
print(df.shape)
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

(45454, 13)
Date range: 2025-01-01 00:27:00 to 2026-08-31 23:59:00


In [2]:
# Time-based split: train on the earliest 80% of incidents, test on the latest 20%.
# A random split would let the model "see the future" during training.
df = df.sort_values("datetime").reset_index(drop=True)
cutoff = df["datetime"].quantile(0.8)

train = df[df["datetime"] < cutoff]
test = df[df["datetime"] >= cutoff]

for name, part in [("Train", train), ("Test", test)]:
    print(f"{name}: {len(part):,} incidents, "
          f"{part['datetime'].min():%Y-%m-%d} to {part['datetime'].max():%Y-%m-%d}, "
          f"long-delay share {part['long_delay'].mean():.1%}")

Train: 36,363 incidents, 2025-01-01 to 2026-05-12, long-delay share 18.9%
Test: 9,091 incidents, 2026-05-12 to 2026-08-31, long-delay share 14.3%


In [3]:
FEATURES = [
    "station_clean", "line_clean", "code_feature", "cause_group",
    "time_period", "hour", "weekday", "month", "is_weekend", "is_rush_hour",
]
TARGET = "long_delay"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [4]:
# Baseline: always predict the most common class (not a long delay)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(confusion_matrix(y_test, y_pred))

Accuracy:  0.857
Precision: 0.000
Recall:    0.000
F1:        0.000
[[7795    0]
 [1296    0]]


Split and baseline:
- Data covers Jan 2025 to Aug 2026. Train: Jan 2025 - May 12, 2026 (36,363 incidents, 18.9% long);
  test: May 12 - Aug 31, 2026 (9,091 incidents, 14.3% long).
- The long-delay share is lower in the test period (possible seasonality or trend), so the test set
  is not distributed exactly like the training set.
- Majority-class baseline: accuracy 0.857 but precision, recall and F1 are all 0. Accuracy is
  misleading here, so models are compared mainly on F1 and recall.